In [3]:
from dotenv import load_dotenv

load_doten

True

Initialising Model

In [7]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-5-mini")

In [8]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)


FileUpload(value=(), accept='.png', description='Upload')

In [9]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [ ]:
from langchain.messages import HumanMessage

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "What Can I cook with this?"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}

])




In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool ("Meal_search", description="From the list of ingredients, search the web for a meal that contains these ingredients")
def meal_search(query: str) -> Dict[str, Any]:
    return tavily_client.search(query)

tools = [meal_search]

In [17]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

config = {"configurable": {"thread_id": "3"}}

system_prompt = """You are a personal chef who helps the user cook with what they already have.

Follow these steps in order:

1. Identify the ingredients.
   - If the user sends a picture, list the ingredients you can see in it.
   - If the user sends a text list, use those ingredients directly.

2. Confirm before searching.
   - If the ingredients come from a picture, show the list and ask the user to confirm
     which ones they want to use. Do not search yet. Wait for their answer.
   - If the user wrote the ingredients themselves, skip this step.

3. Search for recipes.
   - Only after the ingredients are confirmed, use the Meal_search tool.
   - Your search must contain the confirmed ingredients, for example:
     "recipes with eggs, tomatoes, bell peppers and cheese".

4. Answer with 2 recipes. For each recipe, give:
   - the name of the dish
   - the ingredients needed, marking any the user does not have
   - the main steps, in a short numbered list
   - the cooking time

If you cannot see any ingredients in the picture, say so and ask the user to list them."""

chef = create_agent(
    
    model=model,
    checkpointer=InMemorySaver(),
    tools=tools,
    system_prompt=system_prompt

)

response = chef.invoke(
    {"messages": [multimodal_question]},
     config, 
)

print(response['messages'][-1].content)


I can see several ingredients in the photo. Please check the list and tell me which of these you want to use (or add anything I missed) before I search for recipes.

Visible items I can identify:
- Salmon fillets (two pieces)
- Sliced ham / deli meat
- Meat patties (looks like burger or sausage patties)
- Eggs (in door)
- Milk (bottles)
- Yogurt or sour cream (white bowl)
- Grated cheese (container)
- Block(s) of cheese / butter
- Glass jar of chickpeas or beans (jar)
- Small jar of a condiment/preserve (uncertain)
- Broccoli
- Green bell pepper
- Red bell pepper
- Yellow bell pepper
- Carrots
- Cucumber or zucchini (long green)
- Tomatoes (in drawer)
- Apples (in drawer)
- Bottles of juice (door)
- Bottles of water (door)
- Dark bottle (sauce) in door (possibly soy or balsamic)

Anything here you don’t want included? Any other ingredients in your kitchen I should know about (rice, pasta, spices, oil)? Any diet preferences or time limit? Once you confirm which ingredients to use, I’ll 

In [18]:
confirmation = HumanMessage(content="Yes, I want to use eggs, milk, salmom.")
response = chef.invoke(
    {"messages": [confirmation]},
    config,
)
for ___ in response["messages"]:
    ___.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': 'What Can I cook with this?'}, {'type': 'image', 'base64': '...', 'mime_type': 'image/png'}]
================================== Ai Message ==================================

I can see several ingredients in the photo. Please check the list and tell me which of these you want to use (or add anything I missed) before I search for recipes.

Visible items I can identify:
- Salmon fillets (two pieces)
- Sliced ham / deli meat
- Meat patties (looks like burger or sausage patties)
- Eggs (in door)
- Milk (bottles)
- Yogurt or sour cream (white bowl)
- Grated cheese (container)
- Block(s) of cheese / butter
- Glass jar of chickpeas or beans (jar)
- Small jar of a condiment/preserve (uncertain)
- Broccoli
- Green bell pepper
- Red bell pepper
- Yellow bell pepper
- Carrots
- Cucumber or zucchini (long green)
- Tomatoes (in drawer)
- Apples (in drawer)
- Bottles of juice (door)
- Bottles 